In [ ]:
# Русский: установка streamlit и pyngrok
# English: installing streamlit and pyngrok
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 87.8 MB/s eta 0:00:00


In [ ]:
# Русский: аутентификация в ngrok
# English: ngrok authentication

from pyngrok import ngrok
from getpass import getpass

token = getpass("ngrok authtoken: ")
ngrok.set_auth_token(token)

ngrok authtoken: ··········


In [ ]:
%%writefile app.py
import hashlib
import shutil
import time
import zipfile
from collections.abc import Callable
from pathlib import Path
from typing import Literal

import pandas as pd
import streamlit as st
import torch as th
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

try:
    from transformers import AutoImageProcessor, AutoModel, CLIPVisionModelWithProjection, ViTModel
    TRANSFORMERS_AVAILABLE = True
except Exception:
    AutoImageProcessor = None
    AutoModel = None
    CLIPVisionModelWithProjection = None
    ViTModel = None
    TRANSFORMERS_AVAILABLE = False

# Русский: добавляю поддержку собственной ResNet-подобной модели
# любой пользователь может аналогично добавить свои классы модели ниже,
# если хочет загрузить кастомную архитектуру, отсутствующую в torchvision
# English: adding support for a custom ResNet-like model
# English: any user can similarly add their own model classes below
# if they want to load a custom architecture that is not available in torchvision

class ResNet10TIBasicBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: th.Tensor) -> th.Tensor:
        identity = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = out + identity
        out = self.relu(out)
        return out

class ResNet10TI(nn.Module):
    def __init__(self, num_classes: int = 200) -> None:
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.stage1 = ResNet10TIBasicBlock(32, 32, stride=1)
        self.stage2 = ResNet10TIBasicBlock(32, 64, stride=2)
        self.stage3 = ResNet10TIBasicBlock(64, 128, stride=2)
        self.stage4 = ResNet10TIBasicBlock(128, 128, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x: th.Tensor) -> th.Tensor:
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.pool(x)
        x = x.flatten(1)
        x = self.fc(x)
        return x

# Русский: словарь переводов интерфейса
# English: UI translations dictionary
TEXTS = {
    "ru_page_title": "CBIR — оценка сходства изображений",
    "en_page_title": "CBIR — image similarity evaluation",

    "ru_title": "Система оценки сходства изображений для визуального поиска",
    "en_title": "Image similarity evaluation system for visual search",

    "ru_intro": (
        "Загрузите zip-архив с изображениями и выберите retrieval-модель. "
        "Приложение определит структуру датасета, загрузит модель на доступное устройство, "
        "рассчитает глобальные эмбеддинги и позволит выполнить поиск похожих изображений. "
        "Для отдельного изображения можно дополнительно включить reranking по глобальным или патчевым признакам."
    ),
    "en_intro": (
        "Upload an image dataset zip archive and select a retrieval model. "
        "The application detects the dataset structure, loads the model to the available device, "
        "computes global embeddings, and runs similar image search. "
        "For a single query image, reranking can additionally use global or patch-level features."
    ),

    "ru_language_label": "Язык интерфейса",
    "en_language_label": "Interface language",

    "ru_zip_uploader_label": "Zip-архив датасета",
    "en_zip_uploader_label": "Dataset zip archive",

    "ru_model_uploader_label": "Retrieval-модель (.pth / .pt)",
    "en_model_uploader_label": "Retrieval model (.pth / .pt)",

    "ru_retrieval_model_header": "Retrieval-модель",
    "en_retrieval_model_header": "Retrieval model",

    "ru_retrieval_model_source_label": "Источник retrieval-модели",
    "en_retrieval_model_source_label": "Retrieval model source",

    "ru_model_source_upload": "Загрузить .pth / .pt",
    "en_model_source_upload": "Upload .pth / .pt",

    "ru_model_source_recommended": "Выбрать встроенную модель",
    "en_model_source_recommended": "Select a built-in model",

    "ru_recommended_model_select_label": "Модель",
    "en_recommended_model_select_label": "Model",

    "ru_recommended_model_loading": "Загрузка модели: {model_name}...",
    "en_recommended_model_loading": "Loading model: {model_name}...",

    "ru_recommended_preprocess_info": "Для встроенной модели используется стандартный препроцессинг Hugging Face.",
    "en_recommended_preprocess_info": "Built-in models use the standard Hugging Face preprocessing pipeline.",

    "ru_transformers_missing_error": (
        "Для встроенных моделей требуется библиотека transformers. "
        "Установите зависимости: pip install transformers safetensors huggingface_hub"
    ),
    "en_transformers_missing_error": (
        "Built-in models require the transformers library. "
        "Install dependencies with: pip install transformers safetensors huggingface_hub"
    ),

    "ru_uploaded_success": "Датасет загружен и распакован.",
    "en_uploaded_success": "Dataset uploaded and extracted.",

    "ru_model_loaded_success": "Модель загружена.",
    "en_model_loaded_success": "Model loaded.",

    "ru_bad_zip_error": "Файл не является корректным zip-архивом.",
    "en_bad_zip_error": "The uploaded file is not a valid zip archive.",

    "ru_model_load_error": "Не удалось загрузить модель.",
    "en_model_load_error": "Could not load the model.",

    "ru_total_classes_metric": "Классов",
    "en_total_classes_metric": "Classes",

    "ru_total_images_metric": "Изображений",
    "en_total_images_metric": "Images",

    "ru_structure_mode_label": "Режим структуры",
    "en_structure_mode_label": "Structure mode",

    "ru_class_distribution_header": "Распределение изображений по классам",
    "en_class_distribution_header": "Images per class",

    "ru_gallery_header": "Просмотр изображений по классам",
    "en_gallery_header": "Class-based image browser",

    "ru_gallery_header_no_classes": "Просмотр изображений",
    "en_gallery_header_no_classes": "Image browser",

    "ru_select_class_label": "Класс",
    "en_select_class_label": "Class",

    "ru_images_found_in_class": "В классе **{class_name}** найдено изображений: **{count}**.",
    "en_images_found_in_class": "Images in class **{class_name}**: **{count}**.",

    "ru_images_found_total": "Всего найдено изображений: **{count}**.",
    "en_images_found_total": "Total images found: **{count}**.",

    "ru_show_how_many_slider": "Количество изображений в галерее",
    "en_show_how_many_slider": "Number of gallery images",

    "ru_expand_file_list": "Показать список файлов",
    "en_expand_file_list": "Show file list",

    "ru_no_images_found_error": "В архиве не найдено ни одного поддерживаемого изображения.",
    "en_no_images_found_error": "No supported images were found in the archive.",

    "ru_waiting_for_upload_info": (
        "Загрузите zip-архив датасета. После распаковки приложение покажет структуру, "
        "классы и найденные изображения."
    ),
    "en_waiting_for_upload_info": (
        "Upload a dataset zip archive. After extraction, the application will show its structure, "
        "classes, and detected images."
    ),

    "ru_waiting_for_model_info": "Выберите или загрузите retrieval-модель для расчёта эмбеддингов.",
    "en_waiting_for_model_info": "Select or upload a retrieval model to compute embeddings.",

    "ru_could_not_open_image": "Не удалось открыть",
    "en_could_not_open_image": "Could not open",

    "ru_dataset_root_path_label": "Корень датасета",
    "en_dataset_root_path_label": "Dataset root",

    "ru_dataset_technical_info_header": "Техническая информация о датасете",
    "en_dataset_technical_info_header": "Dataset technical information",

    "ru_no_classes_message": (
        "Классы не найдены или найден только один класс. Приложение будет работать в режиме общего списка "
        "изображений; class-based retrieval-метрики будут недоступны."
    ),
    "en_no_classes_message": (
        "No classes were found or only one class was detected. The application will use the global image list mode; "
        "class-based retrieval metrics will be unavailable."
    ),

    "ru_device_metric": "Устройство",
    "en_device_metric": "Device",

    "ru_embedding_dim_metric": "Размерность эмбеддинга",
    "en_embedding_dim_metric": "Embedding dimension",

    "ru_embeddings_ready_success": "Эмбеддинги датасета рассчитаны.",
    "en_embeddings_ready_success": "Dataset embeddings computed.",

    "ru_embeddings_in_progress": "Расчёт эмбеддингов датасета...",
    "en_embeddings_in_progress": "Computing dataset embeddings...",

    "ru_embedding_error": "Не удалось рассчитать эмбеддинги.",
    "en_embedding_error": "Could not compute embeddings.",

    "ru_model_requirements_info": (
        "Для загружаемой .pth/.pt retrieval-модели ожидается full-extractor, сохранённый через torch.save(model, path). "
        "Метод forward должен возвращать тензор эмбеддингов или структуру, из которой можно извлечь первый тензор."
    ),
    "en_model_requirements_info": (
        "An uploaded .pth/.pt retrieval model is expected to be a full extractor saved with torch.save(model, path). "
        "Its forward method should return an embedding tensor or a structure containing a tensor."
    ),

    "ru_query_image": "Изображение-запрос",
    "en_query_image": "Query image",

    "ru_select_query_image_label": "Выберите изображение-запрос",
    "en_select_query_image_label": "Select a query image",

    "ru_find_similar_header": "Поиск похожих изображений",
    "en_find_similar_header": "Similar image search",

    "ru_top_k_slider": "Глубина выдачи top-k",
    "en_top_k_slider": "Top-k retrieval depth",

    "ru_find_similar_button": "Найти похожие изображения",
    "en_find_similar_button": "Find similar images",

    "ru_evaluate_dataset_button": "Оценить весь датасет (retrieval)",
    "en_evaluate_dataset_button": "Evaluate full dataset (retrieval)",

    "ru_full_eval_in_progress": "Полный retrieval-обход датасета...",
    "en_full_eval_in_progress": "Running full-dataset retrieval evaluation...",

    "ru_experiment_results_header": "Итоговые метрики эксперимента",
    "en_experiment_results_header": "Experiment summary metrics",

    "ru_query_results_header": "Метрики по изображениям-запросам",
    "en_query_results_header": "Per-query metrics",

    "ru_download_experiment_csv": "Скачать итоговые метрики CSV",
    "en_download_experiment_csv": "Download summary metrics CSV",

    "ru_download_query_csv": "Скачать per-query метрики CSV",
    "en_download_query_csv": "Download per-query metrics CSV",

    "ru_full_eval_requires_classes": "Полная оценка доступна только для датасета минимум с двумя классами.",
    "en_full_eval_requires_classes": "Full evaluation is available only for datasets with at least two classes.",

    "ru_no_valid_eval_queries": "В датасете нет запросов с другим изображением того же класса.",
    "en_no_valid_eval_queries": "No query has another image from the same class.",

    "ru_press_find_button_info": (
        "Запустите поиск по выбранному изображению или полный retrieval-обход датасета."
    ),
    "en_press_find_button_info": (
        "Run search for the selected query image or full-dataset retrieval evaluation."
    ),

    "ru_relative_path_caption": "Путь: {path}",
    "en_relative_path_caption": "Path: {path}",

    "ru_preprocess_header": "Параметры препроцессинга и инференса",
    "en_preprocess_header": "Preprocessing and inference parameters",

    "ru_image_size_label": "Размер входного изображения для загружаемой retrieval-модели",
    "en_image_size_label": "Input image size for uploaded retrieval model",

    "ru_batch_size_label": "Размер batch",
    "en_batch_size_label": "Batch size",

    "ru_similarity_method_label": "Метод сравнения эмбеддингов",
    "en_similarity_method_label": "Embedding similarity method",

    "ru_similarity_cosine": "Косинусное сходство",
    "en_similarity_cosine": "Cosine similarity",

    "ru_similarity_dot": "Скалярное произведение",
    "en_similarity_dot": "Dot product",

    "ru_similarity_euclidean": "Евклидово расстояние",
    "en_similarity_euclidean": "Euclidean distance",

    "ru_l2_normalize_label": "L2-нормализация эмбеддингов перед сравнением",
    "en_l2_normalize_label": "L2-normalize embeddings before comparison",

    "ru_no_candidate_images_info": "Для выбранного изображения нет других элементов в датасете.",
    "en_no_candidate_images_info": "No other dataset items are available for the selected query image.",

    "ru_error_details_prefix": "Подробности",
    "en_error_details_prefix": "Details",

    "ru_compute_embeddings_button": "Рассчитать эмбеддинги",
    "en_compute_embeddings_button": "Compute embeddings",

    "ru_compute_embeddings_first_info": "Нажмите кнопку, чтобы рассчитать эмбеддинги по датасету.",
    "en_compute_embeddings_first_info": "Press the button to compute dataset embeddings.",

    "ru_total_seconds_metric": "Время расчёта, сек.",
    "en_total_seconds_metric": "Total time, sec.",

    "ru_images_per_second_metric": "Изображений/сек.",
    "en_images_per_second_metric": "Images/sec.",

    "ru_seconds_per_image_metric": "Секунд/изображение",
    "en_seconds_per_image_metric": "Seconds/image",

    "ru_metrics_header": "Retrieval-метрики для выбранного изображения",
    "en_metrics_header": "Retrieval metrics for the selected query image",

    "ru_metrics_unavailable_info": (
        "Retrieval-метрики доступны только для датасета с классами, где релевантность определяется совпадением класса."
    ),
    "en_metrics_unavailable_info": (
        "Retrieval metrics are available only for class-based datasets where relevance is defined by class match."
    ),

    "ru_not_enough_relevant_items_info": (
        "Для выбранного изображения нет других элементов того же класса; retrieval-метрики недоступны."
    ),
    "en_not_enough_relevant_items_info": (
        "The selected query image has no other items from the same class; retrieval metrics are unavailable."
    ),

    "ru_selected_query_not_found_error": "Изображение-запрос не найдено среди рассчитанных эмбеддингов.",
    "en_selected_query_not_found_error": "Selected query image was not found among computed embeddings.",

    "ru_reranking_header": "Reranking для поиска по одному изображению",
    "en_reranking_header": "Reranking for single-query search",

    "ru_reranking_source_label": "Источник reranking-модели",
    "en_reranking_source_label": "Reranking model source",

    "ru_reranking_source_none": "Без reranking",
    "en_reranking_source_none": "No reranking",

    "ru_reranking_model_uploader_label": "Reranking-модель (.pth / .pt)",
    "en_reranking_model_uploader_label": "Reranking model (.pth / .pt)",

    "ru_reranking_feature_type_label": "Признаки для reranking",
    "en_reranking_feature_type_label": "Features for reranking",

    "ru_reranking_feature_global": "Глобальные эмбеддинги",
    "en_reranking_feature_global": "Global embeddings",

    "ru_reranking_feature_patches": "Патчевые эмбеддинги",
    "en_reranking_feature_patches": "Patch embeddings",

    "ru_rerank_top_n_slider": "Количество retrieval-кандидатов для reranking",
    "en_rerank_top_n_slider": "Number of retrieval candidates to rerank",

    "ru_reranking_in_progress": "Reranking top-N кандидатов...",
    "en_reranking_in_progress": "Reranking top-N candidates...",

    "ru_reranking_model_missing_error": "Для выбранного режима reranking нужно загрузить или выбрать reranking-модель.",
    "en_reranking_model_missing_error": "The selected reranking mode requires a reranking model.",

    "ru_patch_embeddings_missing_error": (
        "Выбранная reranking-модель не возвращает patch embeddings. "
        "Выберите reranking по глобальным эмбеддингам или используйте модель с выходом {'global', 'patches'}."
    ),
    "en_patch_embeddings_missing_error": (
        "The selected reranking model does not return patch embeddings. "
        "Choose global-embedding reranking or use a model with {'global', 'patches'} output."
    ),

    "ru_reranking_used_caption": "Использован reranking: {mode}",
    "en_reranking_used_caption": "Reranking used: {mode}",

    "ru_reranking_input_tensor_metric": "Входной тензор reranking-модели",
    "en_reranking_input_tensor_metric": "Reranking input tensor",

    "ru_reranking_output_tensor_metric": "Выходные тензоры reranking-модели",
    "en_reranking_output_tensor_metric": "Reranking output tensors",

    "ru_reranking_tensor_shapes_unavailable": "Не удалось определить формы тензоров reranking-модели.",
    "en_reranking_tensor_shapes_unavailable": "Could not infer reranking model tensor shapes.",
}


# Русский: язык по умолчанию
# English: default language
DEFAULT_LANG: Literal["ru", "en"] = "ru"

# Русский: базовая настройка страницы
# English: initial page configuration
st.set_page_config(
    page_title=TEXTS[f"{DEFAULT_LANG}_page_title"],
    page_icon="🖼️",
    layout="wide",
)

# Русский: поддерживаемые расширения изображений
# English: supported image extensions
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Русский: рабочие пути приложения
# English: application working paths
BASE_DIR = Path.cwd() / "cbir_workspace"
DATASET_DIR = BASE_DIR / "dataset"
MODEL_DIR = BASE_DIR / "model"
EXTRACT_DIR = DATASET_DIR / "extracted"
ZIP_PATH = DATASET_DIR / "uploaded_dataset.zip"
RETRIEVAL_MODEL_PATH = MODEL_DIR / "uploaded_retrieval_model.pth"
RERANKING_MODEL_PATH = MODEL_DIR / "uploaded_reranking_model.pth"

# Русский: устройство для инференса
# English: inference device
DEVICE = th.device("cuda" if th.cuda.is_available() else "cpu")

# Русский: рекомендованные модели с явным извлечением global/patch признаков
# English: recommended models with explicit global/patch feature extraction
RECOMMENDED_MODEL_CONFIGS = {
    "ViT-B/16 — google/vit-base-patch16-224-in21k": {
        "kind": "vit",
        "model_id": "google/vit-base-patch16-224-in21k",
    },
    "ViT-B/32 — google/vit-base-patch32-224-in21k": {
        "kind": "vit",
        "model_id": "google/vit-base-patch32-224-in21k",
    },
    "DINO ViT-S/16 — facebook/dino-vits16": {
        "kind": "vit",
        "model_id": "facebook/dino-vits16",
    },
    "DINO ViT-B/16 — facebook/dino-vitb16": {
        "kind": "vit",
        "model_id": "facebook/dino-vitb16",
    },
    "DINOv2-S/14 — facebook/dinov2-small": {
        "kind": "dinov2",
        "model_id": "facebook/dinov2-small",
    },
    "DINOv2-B/14 — facebook/dinov2-base": {
        "kind": "dinov2",
        "model_id": "facebook/dinov2-base",
    },
    "CLIP ViT-B/16 — openai/clip-vit-base-patch16": {
        "kind": "clip",
        "model_id": "openai/clip-vit-base-patch16",
    },
    "CLIP ViT-B/32 — openai/clip-vit-base-patch32": {
        "kind": "clip",
        "model_id": "openai/clip-vit-base-patch32",
    },
}


# Русский: выбор языка интерфейса
# English: interface language selector
lang = st.selectbox(
    label=TEXTS[f"{DEFAULT_LANG}_language_label"],
    options=["ru", "en"],
    index=0,
)

st.title(TEXTS[f"{lang}_title"])
st.write(TEXTS[f"{lang}_intro"])

device_col, _ = st.columns([1, 3])
device_col.metric(TEXTS[f"{lang}_device_metric"], str(DEVICE))


class ImagePathDataset(Dataset):
    # Русский: датасет путей изображений для батчевого инференса
    # English: image-path dataset for batched inference
    def __init__(
        self,
        image_paths: list[Path],
        transform: Callable[[Image.Image], th.Tensor],
    ) -> None:
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> tuple[th.Tensor, str]:
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert("RGB")
        tensor = self.transform(image)
        return tensor, str(image_path)


class RecommendedViTExtractor(nn.Module):
    # Русский: обёртка ViT/DINO ViT, возвращающая CLS-эмбеддинг и patch-токены
    # English: ViT/DINO ViT wrapper returning the CLS embedding and patch tokens
    def __init__(self, model_id: str) -> None:
        super().__init__()
        self.backbone = ViTModel.from_pretrained(model_id)

    def forward(self, images: th.Tensor) -> dict[str, th.Tensor]:
        outputs = self.backbone(pixel_values=images, return_dict=True)
        tokens = outputs.last_hidden_state
        return {
            "global": tokens[:, 0, :],
            "patches": tokens[:, 1:, :],
        }


class RecommendedDINOv2Extractor(nn.Module):
    # Русский: обёртка DINOv2, учитывающая возможные register-токены
    # English: DINOv2 wrapper with optional register-token handling
    def __init__(self, model_id: str) -> None:
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_id)

    def forward(self, images: th.Tensor) -> dict[str, th.Tensor]:
        outputs = self.backbone(pixel_values=images, return_dict=True)
        tokens = outputs.last_hidden_state
        num_register_tokens = int(getattr(self.backbone.config, "num_register_tokens", 0))
        return {
            "global": tokens[:, 0, :],
            "patches": tokens[:, 1 + num_register_tokens :, :],
        }


class RecommendedCLIPVisionExtractor(nn.Module):
    # Русский: vision-only CLIP, возвращающий спроецированные global и patch признаки
    # English: vision-only CLIP wrapper returning projected global and patch features
    def __init__(self, model_id: str) -> None:
        super().__init__()
        self.backbone = CLIPVisionModelWithProjection.from_pretrained(model_id)

    def forward(self, images: th.Tensor) -> dict[str, th.Tensor]:
        outputs = self.backbone(pixel_values=images, return_dict=True)
        global_embedding = outputs.image_embeds
        patch_embeddings = outputs.last_hidden_state[:, 1:, :]
        patch_embeddings = self.backbone.vision_model.post_layernorm(patch_embeddings)
        patch_embeddings = self.backbone.visual_projection(patch_embeddings)
        return {
            "global": global_embedding,
            "patches": patch_embeddings,
        }


def clear_directory(directory: Path) -> None:
    # Русский: полная очистка рабочей директории
    # English: full cleanup of the working directory
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)


def extract_zip(zip_path: Path, extract_to: Path) -> None:
    # Русский: распаковка zip-архива
    # English: extract zip archive
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_to)


def build_uploaded_file_state_key(uploaded_file) -> tuple[str, int, str]:
    # Русский: ключ состояния файла для отслеживания изменения архива или модели
    # English: file state key for detecting archive or model changes
    file_bytes = uploaded_file.getbuffer()
    file_hash = hashlib.sha256(file_bytes).hexdigest()
    return uploaded_file.name, uploaded_file.size, file_hash


def directory_has_files(directory: Path) -> bool:
    # Русский: проверка, что директория существует и не пуста
    # English: check that directory exists and is not empty
    return directory.exists() and any(directory.iterdir())


def is_hidden_or_system_name(name: str) -> bool:
    # Русский: проверка скрытого или служебного имени
    # English: check hidden or system name
    return name.startswith(".") or name == "__MACOSX"


def is_valid_dir(path: Path) -> bool:
    # Русский: проверка корректной директории
    # English: check valid directory
    return path.is_dir() and not is_hidden_or_system_name(path.name)


def is_valid_image_file(path: Path) -> bool:
    # Русский: проверка корректного файла изображения
    # English: check valid image file
    return (
        path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
        and not path.name.startswith(".")
        and not path.name.startswith("._")
    )


def count_images_recursively_in_directory(directory: Path) -> int:
    # Русский: рекурсивный подсчёт изображений
    # English: recursively count images
    return sum(1 for p in directory.rglob("*") if is_valid_image_file(p))


def collect_all_images_recursively(directory: Path) -> list[Path]:
    # Русский: рекурсивный сбор всех изображений
    # English: recursively collect all images
    return sorted([p for p in directory.rglob("*") if is_valid_image_file(p)])


def detect_dataset_structure(extract_dir: Path) -> tuple[Path | None, str]:
    # Русский: определение режима структуры датасета
    # English: detect dataset structure mode
    direct_subdirs = [p for p in extract_dir.iterdir() if is_valid_dir(p)]

    if len(direct_subdirs) == 1:
        dataset_name_dir = direct_subdirs[0]
        inner_subdirs = [p for p in dataset_name_dir.iterdir() if is_valid_dir(p)]
        inner_class_dirs = [
            p for p in inner_subdirs
            if count_images_recursively_in_directory(p) > 0
        ]

        if len(inner_class_dirs) > 1:
            return dataset_name_dir, "dataset_with_classes"

    direct_class_dirs = [
        p for p in direct_subdirs
        if count_images_recursively_in_directory(p) > 0
    ]

    if len(direct_class_dirs) > 1:
        return extract_dir, "direct_classes"

    return None, "no_classes"


def build_class_dataset_index(dataset_root: Path) -> dict[str, list[Path]]:
    # Русский: индекс изображений по классам
    # English: class-based image index
    dataset_index: dict[str, list[Path]] = {}

    for class_dir in sorted(dataset_root.iterdir()):
        if not is_valid_dir(class_dir):
            continue

        image_paths = sorted([
            p for p in class_dir.rglob("*")
            if is_valid_image_file(p)
        ])

        if len(image_paths) > 0:
            dataset_index[class_dir.name] = image_paths

    return dataset_index


def build_global_image_index(extract_dir: Path) -> dict[str, list[Path]]:
    # Русский: общий индекс изображений без классов
    # English: global image index without classes
    all_images = collect_all_images_recursively(extract_dir)
    return {"all_images": all_images}


def flatten_dataset_index(dataset_index: dict[str, list[Path]]) -> list[Path]:
    # Русский: разворачивание индекса в единый список путей
    # English: flattening the index into a single path list
    all_paths: list[Path] = []
    for paths in dataset_index.values():
        all_paths.extend(paths)
    return all_paths


def build_path_to_label(dataset_index: dict[str, list[Path]], classes_found: bool) -> dict[Path, str | None]:
    # Русский: построение словаря путь -> метка класса
    # English: build path -> class label mapping
    path_to_label: dict[Path, str | None] = {}

    if not classes_found:
        for paths in dataset_index.values():
            for path in paths:
                path_to_label[path] = None
        return path_to_label

    for class_name, paths in dataset_index.items():
        for path in paths:
            path_to_label[path] = class_name

    return path_to_label


def load_image_safely(image_path: Path) -> Image.Image | None:
    # Русский: безопасная загрузка изображения
    # English: safe image loading
    try:
        image = Image.open(image_path)
        image.load()
        return image
    except Exception:
        return None


def structure_mode_to_text(mode: str) -> str:
    # Русский: текстовое описание режима структуры
    # English: text representation of structure mode
    if mode == "dataset_with_classes":
        return "extract_dir / dataset_name / class_1 / class_2"
    if mode == "direct_classes":
        return "extract_dir / class_1 / class_2"
    return "no classes / single class"


def relative_path_for_display(path: Path, root: Path) -> str:
    # Русский: представление пути относительно корня датасета
    # English: display path relative to dataset root
    try:
        return str(path.relative_to(root))
    except Exception:
        return str(path)


def build_preprocess(image_size: int) -> transforms.Compose:
    # Русский: стандартный ImageNet-препроцессинг для загружаемых моделей
    # English: standard ImageNet preprocessing for uploaded models
    return transforms.Compose([
        transforms.Resize(image_size),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])


def get_processor_size_value(size_object, default_value: int = 224) -> int:
    # Русский: извлечение одного размера из HF processor.size/crop_size
    # English: extract one integer size from HF processor.size/crop_size
    if isinstance(size_object, int):
        return size_object

    if isinstance(size_object, dict):
        for key in ["height", "shortest_edge", "width"]:
            if key in size_object and isinstance(size_object[key], int):
                return size_object[key]

    return default_value


@st.cache_resource(show_spinner=False)
def build_recommended_preprocess_cached(model_label: str) -> transforms.Compose:
    # Русский: построение препроцессинга из Hugging Face processor выбранной модели
    # English: build preprocessing from the selected model's Hugging Face processor
    if not TRANSFORMERS_AVAILABLE:
        raise ImportError("transformers is not installed")

    model_id = RECOMMENDED_MODEL_CONFIGS[model_label]["model_id"]
    processor = AutoImageProcessor.from_pretrained(model_id)

    resize_size = get_processor_size_value(getattr(processor, "size", None), default_value=224)
    crop_size = get_processor_size_value(getattr(processor, "crop_size", None), default_value=resize_size)
    image_mean = getattr(processor, "image_mean", [0.485, 0.456, 0.406])
    image_std = getattr(processor, "image_std", [0.229, 0.224, 0.225])

    return transforms.Compose([
        transforms.Resize(resize_size),
        transforms.CenterCrop(crop_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=image_mean, std=image_std),
    ])


def format_tensor_shape(shape: tuple[int, ...]) -> str:
    # Русский: форматирование формы тензора для интерфейса
    # English: format tensor shape for UI display
    return " × ".join(str(dim) for dim in shape)


def replace_first_dim(shape: tuple[int, ...], batch_size: int) -> tuple[int, ...]:
    # Русский: замена batch-размерности на выбранный batch size
    # English: replace the batch dimension with the selected batch size
    if len(shape) == 0:
        return shape
    return (batch_size,) + shape[1:]


def get_first_tensor_from_output(output: object) -> th.Tensor:
    # Русский: извлечение первого тензора из выхода пользовательской модели
    # English: extract the first tensor from a custom model output
    if isinstance(output, th.Tensor):
        return output

    if isinstance(output, dict):
        for value in output.values():
            if isinstance(value, th.Tensor):
                return value

    if isinstance(output, (list, tuple)):
        for value in output:
            if isinstance(value, th.Tensor):
                return value

    if hasattr(output, "image_embeds") and isinstance(output.image_embeds, th.Tensor):
        return output.image_embeds

    if hasattr(output, "pooler_output") and isinstance(output.pooler_output, th.Tensor):
        return output.pooler_output

    if hasattr(output, "last_hidden_state") and isinstance(output.last_hidden_state, th.Tensor):
        return output.last_hidden_state[:, 0, :]

    raise TypeError("Model output does not contain a tensor.")


def get_global_tensor_from_output(output: object) -> th.Tensor:
    # Русский: извлечение глобального эмбеддинга с fallback на старую логику
    # English: extract global embedding with fallback to the legacy logic
    if isinstance(output, dict) and isinstance(output.get("global"), th.Tensor):
        return output["global"]

    return get_first_tensor_from_output(output)


def get_patch_tensor_from_output(output: object) -> th.Tensor | None:
    # Русский: извлечение patch embeddings, если модель явно их возвращает
    # English: extract patch embeddings if the model explicitly returns them
    if isinstance(output, dict) and isinstance(output.get("patches"), th.Tensor):
        return output["patches"]

    if hasattr(output, "last_hidden_state") and isinstance(output.last_hidden_state, th.Tensor):
        return output.last_hidden_state[:, 1:, :]

    return None


def normalize_global_embedding_shape(embeddings: th.Tensor) -> th.Tensor:
    # Русский: приведение глобальных эмбеддингов к форме [B, D]
    # English: reshape global embeddings to [B, D]
    if embeddings.ndim == 1:
        return embeddings.unsqueeze(0)
    if embeddings.ndim > 2:
        return th.flatten(embeddings, start_dim=1)
    return embeddings


def normalize_patch_embedding_shape(patches: th.Tensor) -> th.Tensor:
    # Русский: приведение patch embeddings к форме [B, N, D]
    # English: reshape patch embeddings to [B, N, D]
    if patches.ndim == 2:
        return patches.unsqueeze(0)

    if patches.ndim == 4:
        return patches.flatten(2).transpose(1, 2)

    return patches


def infer_model_tensor_shapes(
    model: nn.Module,
    preprocess: Callable[[Image.Image], th.Tensor],
    batch_size: int,
    device: th.device,
) -> tuple[tuple[int, ...], dict[str, tuple[int, ...]]]:
    # Русский: определение форм входного и выходных тензоров модели через dummy-forward
    # English: infer input and output tensor shapes with a dummy forward pass
    dummy_image = Image.new("RGB", (512, 512))
    single_input_tensor = preprocess(dummy_image)
    input_shape = (batch_size,) + tuple(single_input_tensor.shape)

    dummy_batch = single_input_tensor.unsqueeze(0).to(device)

    with th.no_grad():
        output = model(dummy_batch)

    output_shapes: dict[str, tuple[int, ...]] = {}

    if isinstance(output, th.Tensor):
        output_shapes["output"] = replace_first_dim(tuple(output.shape), batch_size)
        return input_shape, output_shapes

    if isinstance(output, dict):
        for key, value in output.items():
            if isinstance(value, th.Tensor):
                output_shapes[str(key)] = replace_first_dim(tuple(value.shape), batch_size)

        if len(output_shapes) > 0:
            return input_shape, output_shapes

    if isinstance(output, (list, tuple)):
        for idx, value in enumerate(output):
            if isinstance(value, th.Tensor):
                output_shapes[f"output_{idx}"] = replace_first_dim(tuple(value.shape), batch_size)

        if len(output_shapes) > 0:
            return input_shape, output_shapes

    if hasattr(output, "image_embeds") and isinstance(output.image_embeds, th.Tensor):
        output_shapes["image_embeds"] = replace_first_dim(tuple(output.image_embeds.shape), batch_size)

    if hasattr(output, "pooler_output") and isinstance(output.pooler_output, th.Tensor):
        output_shapes["pooler_output"] = replace_first_dim(tuple(output.pooler_output.shape), batch_size)

    if hasattr(output, "last_hidden_state") and isinstance(output.last_hidden_state, th.Tensor):
        output_shapes["last_hidden_state"] = replace_first_dim(tuple(output.last_hidden_state.shape), batch_size)

    if len(output_shapes) == 0:
        raise TypeError("Model output does not contain tensors.")

    return input_shape, output_shapes


def build_recommended_model(model_label: str) -> nn.Module:
    # Русский: создание рекомендованной модели по названию из списка
    # English: build a recommended model by its label
    if not TRANSFORMERS_AVAILABLE:
        raise ImportError("transformers is not installed")

    config = RECOMMENDED_MODEL_CONFIGS[model_label]
    kind = config["kind"]
    model_id = config["model_id"]

    if kind == "vit":
        return RecommendedViTExtractor(model_id=model_id)

    if kind == "dinov2":
        return RecommendedDINOv2Extractor(model_id=model_id)

    if kind == "clip":
        return RecommendedCLIPVisionExtractor(model_id=model_id)

    raise ValueError(f"Unsupported recommended model kind: {kind}")


@st.cache_resource(show_spinner=False)
def load_recommended_model_cached(model_label: str, device_string: str) -> nn.Module:
    # Русский: кэшированная загрузка рекомендованной модели, чтобы не скачивать её при каждом rerun
    # English: cached recommended model loading to avoid re-downloading on every rerun
    model = build_recommended_model(model_label)
    device = th.device(device_string)
    model = model.to(device)
    model.eval()
    return model


def load_uploaded_model(uploaded_file, model_path: Path, device: th.device) -> nn.Module:
    # Русский: сохранение и загрузка пользовательской full-model .pth/.pt модели
    # English: save and load a user-provided full-model .pth/.pt file
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    with open(model_path, "wb") as f:
        f.write(uploaded_file.getbuffer())

    model = th.load  (model_path, weights_only=False, map_location=device)
    model = model.to(device)
    model.eval()
    return model


def compute_dataset_embeddings(
    model: nn.Module,
    image_paths: list[Path],
    preprocess: Callable[[Image.Image], th.Tensor],
    batch_size: int,
    device: th.device,
) -> tuple[th.Tensor, list[Path]]:
    # Русский: расчёт только глобальных эмбеддингов по всему датасету
    # English: compute only global embeddings for the whole dataset
    dataset = ImagePathDataset(image_paths=image_paths, transform=preprocess)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=th.cuda.is_available(),
    )

    all_embeddings = []
    all_paths: list[Path] = []

    with th.no_grad():
        for batch_images, batch_paths in dataloader:
            batch_images = batch_images.to(device)
            output = model(batch_images)
            embeddings = get_global_tensor_from_output(output)
            embeddings = normalize_global_embedding_shape(embeddings)

            all_embeddings.append(embeddings.detach().cpu())
            all_paths.extend([Path(p) for p in batch_paths])

    embeddings_matrix = th.cat(all_embeddings, dim=0)
    return embeddings_matrix, all_paths


def extract_features_for_paths(
    model: nn.Module,
    image_paths: list[Path],
    preprocess: Callable[[Image.Image], th.Tensor],
    batch_size: int,
    device: th.device,
) -> tuple[th.Tensor, th.Tensor | None, list[Path]]:
    # Русский: извлечение global/patch признаков только для query и top-N кандидатов reranking
    # English: extract global/patch features only for query and top-N reranking candidates
    dataset = ImagePathDataset(image_paths=image_paths, transform=preprocess)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=th.cuda.is_available(),
    )

    all_globals = []
    all_patches = []
    all_paths: list[Path] = []
    patches_available = True

    with th.no_grad():
        for batch_images, batch_paths in dataloader:
            batch_images = batch_images.to(device)
            output = model(batch_images)

            global_embeddings = get_global_tensor_from_output(output)
            global_embeddings = normalize_global_embedding_shape(global_embeddings)
            all_globals.append(global_embeddings.detach().cpu())

            patch_embeddings = get_patch_tensor_from_output(output)
            if patch_embeddings is None:
                patches_available = False
            elif patches_available:
                patch_embeddings = normalize_patch_embedding_shape(patch_embeddings)
                all_patches.append(patch_embeddings.detach().cpu())

            all_paths.extend([Path(p) for p in batch_paths])

    global_matrix = th.cat(all_globals, dim=0)
    patch_tensor = th.cat(all_patches, dim=0) if patches_available and len(all_patches) > 0 else None
    return global_matrix, patch_tensor, all_paths


def prepare_embeddings_for_search(
    embeddings_matrix: th.Tensor,
    similarity_method: str,
    l2_normalize_before_search: bool,
) -> th.Tensor:
    # Русский: подготовка эмбеддингов к выбранному способу сравнения
    # English: prepare embeddings for the selected comparison method
    if similarity_method == "cosine":
        return F.normalize(embeddings_matrix, dim=1)

    if l2_normalize_before_search:
        return F.normalize(embeddings_matrix, dim=1)

    return embeddings_matrix


def compute_similarity_scores(
    embeddings_matrix: th.Tensor,
    query_index: int,
    similarity_method: str,
    l2_normalize_before_search: bool,
) -> tuple[th.Tensor, bool, str]:
    # Русский: универсальный расчёт score между query и всеми эмбеддингами
    # English: universal score computation between query and all embeddings
    search_embeddings = prepare_embeddings_for_search(
        embeddings_matrix=embeddings_matrix,
        similarity_method=similarity_method,
        l2_normalize_before_search=l2_normalize_before_search,
    )

    query_embedding = search_embeddings[query_index].unsqueeze(0)

    if similarity_method == "cosine":
        scores = search_embeddings @ query_embedding.squeeze(0)
        larger_is_better = True
        metric_name = "CosSim"

    elif similarity_method == "dot":
        scores = search_embeddings @ query_embedding.squeeze(0)
        larger_is_better = True
        metric_name = "Dot"

    elif similarity_method == "euclidean":
        scores = th.cdist(query_embedding, search_embeddings, p=2).squeeze(0)
        larger_is_better = False
        metric_name = "L2Dist"

    else:
        raise ValueError(f"Unknown similarity method: {similarity_method}")

    return scores, larger_is_better, metric_name


def rank_all_candidate_images(
    embeddings_matrix: th.Tensor,
    query_index: int,
    similarity_method: str,
    l2_normalize_before_search: bool,
) -> tuple[th.Tensor, th.Tensor, str]:
    # Русский: ранжирование всех изображений датасета кроме query
    # English: rank all dataset images except the query
    scores, larger_is_better, metric_name = compute_similarity_scores(
        embeddings_matrix=embeddings_matrix,
        query_index=query_index,
        similarity_method=similarity_method,
        l2_normalize_before_search=l2_normalize_before_search,
    )

    candidate_indices = th.arange(scores.shape[0])
    candidate_indices = candidate_indices[candidate_indices != query_index]
    candidate_scores = scores[candidate_indices]
    order = th.argsort(candidate_scores, descending=larger_is_better)

    ranked_indices = candidate_indices[order]
    ranked_scores = candidate_scores[order]
    return ranked_indices, ranked_scores, metric_name


def find_similar_images(
    embeddings_matrix: th.Tensor,
    query_index: int,
    top_k: int,
    similarity_method: str,
    l2_normalize_before_search: bool,
) -> tuple[th.Tensor, th.Tensor, str]:
    # Русский: поиск top-k похожих изображений выбранным методом сравнения
    # English: search top-k similar images with the selected comparison method
    ranked_indices, ranked_scores, metric_name = rank_all_candidate_images(
        embeddings_matrix=embeddings_matrix,
        query_index=query_index,
        similarity_method=similarity_method,
        l2_normalize_before_search=l2_normalize_before_search,
    )

    top_k = min(top_k, len(ranked_indices))
    return ranked_indices[:top_k], ranked_scores[:top_k], metric_name


def compute_global_rerank_scores(
    query_embedding: th.Tensor,
    candidate_embeddings: th.Tensor,
    similarity_method: str,
    l2_normalize_before_search: bool,
) -> tuple[th.Tensor, bool, str]:
    # Русский: расчёт score для global reranking только по top-N кандидатам
    # English: compute global reranking scores only for top-N candidates
    all_embeddings = th.cat([query_embedding.unsqueeze(0), candidate_embeddings], dim=0)
    prepared_embeddings = prepare_embeddings_for_search(
        embeddings_matrix=all_embeddings,
        similarity_method=similarity_method,
        l2_normalize_before_search=l2_normalize_before_search,
    )

    query = prepared_embeddings[0].unsqueeze(0)
    candidates = prepared_embeddings[1:]

    if similarity_method == "cosine":
        scores = candidates @ query.squeeze(0)
        return scores, True, "RerankCosSim"

    if similarity_method == "dot":
        scores = candidates @ query.squeeze(0)
        return scores, True, "RerankDot"

    if similarity_method == "euclidean":
        scores = th.cdist(query, candidates, p=2).squeeze(0)
        return scores, False, "RerankL2Dist"

    raise ValueError(f"Unknown similarity method: {similarity_method}")


def compute_patch_maxsim_scores(
    query_patches: th.Tensor,
    candidate_patches: th.Tensor,
) -> th.Tensor:
    # Русский: явный расчёт symmetric MaxSim
    # English: explicit symmetric MaxSim computation
    query_patches = F.normalize(query_patches, dim=-1)
    candidate_patches = F.normalize(candidate_patches, dim=-1)

    candidate_scores = []
    for current_candidate_patches in candidate_patches:
        similarity_matrix = query_patches @ current_candidate_patches.T
        query_to_candidate = similarity_matrix.max(dim=1).values.mean()
        candidate_to_query = similarity_matrix.max(dim=0).values.mean()
        score = 0.5 * (query_to_candidate + candidate_to_query)
        candidate_scores.append(score)

    return th.stack(candidate_scores)


def rerank_candidate_indices(
    reranking_model: nn.Module,
    reranking_preprocess: Callable[[Image.Image], th.Tensor],
    query_path: Path,
    candidate_indices: th.Tensor,
    embedding_paths: list[Path],
    batch_size: int,
    device: th.device,
    reranking_feature_type: str,
    similarity_method: str,
    l2_normalize_before_search: bool,
) -> tuple[th.Tensor, th.Tensor, str]:
    # Русский: переупорядочивание retrieval-кандидатов global или patch признаками
    # English: rerank retrieval candidates using global or patch features
    candidate_paths = [embedding_paths[idx] for idx in candidate_indices.tolist()]
    reranking_paths = [query_path] + candidate_paths

    global_features, patch_features, _ = extract_features_for_paths(
        model=reranking_model,
        image_paths=reranking_paths,
        preprocess=reranking_preprocess,
        batch_size=batch_size,
        device=device,
    )

    if reranking_feature_type == "global":
        scores, larger_is_better, metric_name = compute_global_rerank_scores(
            query_embedding=global_features[0],
            candidate_embeddings=global_features[1:],
            similarity_method=similarity_method,
            l2_normalize_before_search=l2_normalize_before_search,
        )
    elif reranking_feature_type == "patches":
        if patch_features is None:
            raise ValueError("Patch embeddings are not available for the selected reranking model.")
        scores = compute_patch_maxsim_scores(
            query_patches=patch_features[0],
            candidate_patches=patch_features[1:],
        )
        larger_is_better = True
        metric_name = "PatchMaxSim"
    else:
        raise ValueError(f"Unknown reranking feature type: {reranking_feature_type}")

    order = th.argsort(scores, descending=larger_is_better)
    reranked_indices = candidate_indices[order]
    reranked_scores = scores[order]
    return reranked_indices, reranked_scores, metric_name


def compute_query_metrics(
    embeddings_matrix: th.Tensor,
    embedding_paths: list[Path],
    query_index: int,
    query_path: Path,
    path_to_label: dict[Path, str | None],
    top_k: int,
    similarity_method: str,
    l2_normalize_before_search: bool,
) -> dict[str, int | float | None] | None:
    # Русский: расчёт retrieval-метрик для одного изображения-запроса по полному retrieval-ранжированию
    # English: compute retrieval metrics for one query using the full retrieval ranking
    query_label = path_to_label.get(query_path)
    if query_label is None:
        return None

    ranked_indices, _, _ = rank_all_candidate_images(
        embeddings_matrix=embeddings_matrix,
        query_index=query_index,
        similarity_method=similarity_method,
        l2_normalize_before_search=l2_normalize_before_search,
    )

    return compute_metrics_from_ranked_indices(
        ranked_indices=ranked_indices,
        embedding_paths=embedding_paths,
        query_path=query_path,
        path_to_label=path_to_label,
        top_k=top_k,
    )


def compute_metrics_from_ranked_indices(
    ranked_indices: th.Tensor,
    embedding_paths: list[Path],
    query_path: Path,
    path_to_label: dict[Path, str | None],
    top_k: int,
) -> dict[str, int | float | None] | None:
    # Русский: расчёт метрик по готовому списку индексов, AP считается по полному переданному списку
    # English: compute metrics from prepared ranked indices; AP is computed over the full provided list
    query_label = path_to_label.get(query_path)
    if query_label is None:
        return None

    relevance = []
    for idx in ranked_indices.tolist():
        current_path = embedding_paths[idx]
        relevance.append(1 if path_to_label.get(current_path) == query_label else 0)

    relevant_total = sum(
        1
        for current_path in embedding_paths
        if current_path != query_path and path_to_label.get(current_path) == query_label
    )

    return compute_metrics_from_relevance(
        relevance=relevance,
        top_k=top_k,
        relevant_total=relevant_total,
    )


def compute_metrics_from_relevance(
    relevance: list[int],
    top_k: int,
    relevant_total: int | None = None,
) -> dict[str, int | float | None]:
    # Русский: общий расчёт P@k, R@k, P@1, HitRate@k и AP по бинарной релевантности
    # English: common P@k, R@k, P@1, HitRate@k and AP computation from binary relevance
    if relevant_total is None:
        relevant_total = sum(relevance)

    if relevant_total == 0:
        return {
            "relevant_total": 0,
            "precision_at_k": None,
            "recall_at_k": None,
            "precision_at_1": None,
            "hit_rate_at_k": None,
            "ap": None,
        }

    effective_k = min(top_k, len(relevance))
    top_relevance = relevance[:effective_k]
    relevant_in_top_k = sum(top_relevance)

    precision_at_k = relevant_in_top_k / effective_k if effective_k > 0 else 0.0
    recall_at_k = relevant_in_top_k / relevant_total
    precision_at_1 = float(top_relevance[0]) if len(top_relevance) > 0 else 0.0
    hit_rate_at_k = float(relevant_in_top_k > 0)

    precision_sum = 0.0
    hits_so_far = 0
    for rank, rel in enumerate(relevance, start=1):
        if rel == 1:
            hits_so_far += 1
            precision_sum += hits_so_far / rank

    ap = precision_sum / relevant_total

    return {
        "relevant_total": relevant_total,
        "precision_at_k": precision_at_k,
        "recall_at_k": recall_at_k,
        "precision_at_1": precision_at_1,
        "hit_rate_at_k": hit_rate_at_k,
        "ap": ap,
    }


def evaluate_full_dataset(
    embeddings_matrix: th.Tensor,
    embedding_paths: list[Path],
    path_to_label: dict[Path, str | None],
    top_k: int,
    similarity_method: str,
    l2_normalize_before_search: bool,
    model_name: str,
    image_size: int | str,
    batch_size: int,
    display_root: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Русский: выполняет полный обход датасета только по retrieval-выдаче
    # English: runs full-dataset evaluation only with retrieval results
    p_at_k_column = f"P@{top_k}"
    r_at_k_column = f"R@{top_k}"
    hit_rate_at_k_column = f"HitRate@{top_k}"

    query_rows: list[dict] = []

    for query_index, query_path in enumerate(embedding_paths):
        query_label = path_to_label.get(query_path)
        if query_label is None:
            continue

        metrics = compute_query_metrics(
            embeddings_matrix=embeddings_matrix,
            embedding_paths=embedding_paths,
            query_index=query_index,
            query_path=query_path,
            path_to_label=path_to_label,
            top_k=top_k,
            similarity_method=similarity_method,
            l2_normalize_before_search=l2_normalize_before_search,
        )

        if metrics is None or metrics["relevant_total"] == 0:
            query_rows.append({
                "query_path": relative_path_for_display(query_path, display_root),
                "class": query_label,
                "AP": None,
                "P@1": None,
                p_at_k_column: None,
                r_at_k_column: None,
                hit_rate_at_k_column: None,
            })
            continue

        query_rows.append({
            "query_path": relative_path_for_display(query_path, display_root),
            "class": query_label,
            "AP": metrics["ap"],
            "P@1": metrics["precision_at_1"],
            p_at_k_column: metrics["precision_at_k"],
            r_at_k_column: metrics["recall_at_k"],
            hit_rate_at_k_column: metrics["hit_rate_at_k"],
        })

    query_columns = [
        "query_path",
        "class",
        "AP",
        "P@1",
        p_at_k_column,
        r_at_k_column,
        hit_rate_at_k_column,
    ]
    query_df = pd.DataFrame(query_rows, columns=query_columns)

    valid_query_df = query_df[query_df["AP"].notna()] if not query_df.empty else query_df

    if valid_query_df.empty:
        summary_df = pd.DataFrame([{
            "model_name": model_name,
            "image_size": image_size,
            "batch_size": batch_size,
            "similarity": similarity_method,
            "normalize": l2_normalize_before_search,
            "P@1": None,
            p_at_k_column: None,
            r_at_k_column: None,
            hit_rate_at_k_column: None,
            "mAP": None,
        }])
    else:
        summary_df = pd.DataFrame([{
            "model_name": model_name,
            "image_size": image_size,
            "batch_size": batch_size,
            "similarity": similarity_method,
            "normalize": l2_normalize_before_search,
            "P@1": valid_query_df["P@1"].mean(),
            p_at_k_column: valid_query_df[p_at_k_column].mean(),
            r_at_k_column: valid_query_df[r_at_k_column].mean(),
            hit_rate_at_k_column: valid_query_df[hit_rate_at_k_column].mean(),
            "mAP": valid_query_df["AP"].mean(),
        }])

    query_df = query_df[query_columns]
    query_df = query_df.sort_values(["AP", p_at_k_column], ascending=True, na_position="last")

    return summary_df, query_df


def dataframe_to_csv_bytes(dataframe: pd.DataFrame) -> bytes:
    # Русский: CSV в кодировке UTF-8 with BOM удобнее открывать в Excel
    # English: UTF-8 with BOM is more convenient when opening CSV files in Excel
    return dataframe.to_csv(index=False).encode("utf-8-sig")


# Русский: инициализация session state для датасета, эмбеддингов и результатов поиска
# English: initialize session state for dataset, embeddings and search results
if "dataset_state_key" not in st.session_state:
    st.session_state["dataset_state_key"] = None
if "embedding_result" not in st.session_state:
    st.session_state["embedding_result"] = None
if "embedding_state_key" not in st.session_state:
    st.session_state["embedding_state_key"] = None
if "search_result" not in st.session_state:
    st.session_state["search_result"] = None
if "full_eval_result" not in st.session_state:
    st.session_state["full_eval_result"] = None
if "active_result_type" not in st.session_state:
    st.session_state["active_result_type"] = None


# Русский: загрузка zip-архива датасета
# English: dataset zip uploader
uploaded_zip = st.file_uploader(
    TEXTS[f"{lang}_zip_uploader_label"],
    type=["zip"],
)

if uploaded_zip is None:
    st.info(TEXTS[f"{lang}_waiting_for_upload_info"])
    st.stop()


# Русский: сохранение и распаковка датасета только при изменении архива
# English: save and extract dataset only when the archive changes
dataset_state_key = build_uploaded_file_state_key(uploaded_zip)
dataset_changed = st.session_state.get("dataset_state_key") != dataset_state_key
dataset_missing_on_disk = not directory_has_files(EXTRACT_DIR)

if dataset_changed or dataset_missing_on_disk:
    clear_directory(DATASET_DIR)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    with open(ZIP_PATH, "wb") as f:
        f.write(uploaded_zip.getbuffer())

    try:
        extract_zip(ZIP_PATH, EXTRACT_DIR)
    except zipfile.BadZipFile:
        st.error(TEXTS[f"{lang}_bad_zip_error"])
        st.stop()

    st.session_state["dataset_state_key"] = dataset_state_key
    st.session_state["embedding_result"] = None
    st.session_state["embedding_state_key"] = None
    st.session_state["search_result"] = None
    st.session_state["full_eval_result"] = None
    st.session_state["active_result_type"] = None


# Русский: определение структуры датасета
# English: detect dataset structure
dataset_root, structure_mode = detect_dataset_structure(EXTRACT_DIR)

# Русский: индекс по классам или общий индекс
# English: class index or global index
if structure_mode in {"dataset_with_classes", "direct_classes"} and dataset_root is not None:
    dataset_index = build_class_dataset_index(dataset_root)
    classes_found = True
else:
    dataset_index = build_global_image_index(EXTRACT_DIR)
    classes_found = False


# Русский: проверка наличия изображений
# English: check that images exist
total_images = sum(len(paths) for paths in dataset_index.values())
if total_images == 0:
    st.error(TEXTS[f"{lang}_no_images_found_error"])
    st.stop()


total_classes = len(dataset_index) if classes_found else 0
all_dataset_paths = flatten_dataset_index(dataset_index)
display_root = dataset_root if dataset_root is not None else EXTRACT_DIR
path_to_label = build_path_to_label(dataset_index, classes_found)


# Русский: общая статистика по датасету
# English: dataset summary statistics
st.success(TEXTS[f"{lang}_uploaded_success"])
col1, col2 = st.columns(2)
col1.metric(TEXTS[f"{lang}_total_classes_metric"], total_classes)
col2.metric(TEXTS[f"{lang}_total_images_metric"], total_images)

with st.expander(TEXTS[f"{lang}_dataset_technical_info_header"]):
    st.write(
        f"**{TEXTS[f'{lang}_structure_mode_label']}:** "
        f"`{structure_mode_to_text(structure_mode)}`"
    )
    if dataset_root is not None:
        st.write(f"**{TEXTS[f'{lang}_dataset_root_path_label']}:** `{dataset_root}`")


# Русский: распределение по классам
# English: class distribution
if classes_found:
    st.subheader(TEXTS[f"{lang}_class_distribution_header"])
    for class_name, class_image_paths in dataset_index.items():
        st.write(f"**{class_name}** — {len(class_image_paths)}")
else:
    st.info(TEXTS[f"{lang}_no_classes_message"])


# Русский: просмотр изображений
# English: image browsing
if classes_found:
    st.subheader(TEXTS[f"{lang}_gallery_header"])
    class_names = list(dataset_index.keys())
    selected_class = st.selectbox(
        TEXTS[f"{lang}_select_class_label"],
        class_names,
    )
    image_paths = dataset_index[selected_class]
    st.write(
        TEXTS[f"{lang}_images_found_in_class"].format(
            class_name=selected_class,
            count=len(image_paths),
        )
    )
else:
    st.subheader(TEXTS[f"{lang}_gallery_header_no_classes"])
    bucket_name = list(dataset_index.keys())[0]
    image_paths = dataset_index[bucket_name]
    st.write(TEXTS[f"{lang}_images_found_total"].format(count=len(image_paths)))


max_images_to_show = st.slider(
    TEXTS[f"{lang}_show_how_many_slider"],
    min_value=1,
    max_value=len(image_paths),
    value=min(10, len(image_paths)),
)

selected_image_paths = image_paths[:max_images_to_show]
columns = st.columns(4)

for idx, image_path in enumerate(selected_image_paths):
    image = load_image_safely(image_path)
    with columns[idx % 4]:
        if image is not None:
            st.image(
                image,
                caption=relative_path_for_display(image_path, display_root),
                use_container_width=True,
            )
        else:
            st.warning(f"{TEXTS[f'{lang}_could_not_open_image']}: {image_path.name}")

with st.expander(TEXTS[f"{lang}_expand_file_list"]):
    for path in image_paths:
        st.write(relative_path_for_display(path, display_root))


# Русский: выбор и загрузка retrieval-модели
# English: retrieval model selection and loading
st.subheader(TEXTS[f"{lang}_retrieval_model_header"])

retrieval_source_label_to_value = {
    TEXTS[f"{lang}_model_source_upload"]: "upload",
    TEXTS[f"{lang}_model_source_recommended"]: "recommended",
}

selected_retrieval_source_label = st.selectbox(
    TEXTS[f"{lang}_retrieval_model_source_label"],
    list(retrieval_source_label_to_value.keys()),
)
retrieval_source = retrieval_source_label_to_value[selected_retrieval_source_label]

uploaded_retrieval_model = None
selected_retrieval_recommended_label = None
retrieval_model_name = ""
retrieval_model_state_key = None

if retrieval_source == "upload":
    uploaded_retrieval_model = st.file_uploader(
        TEXTS[f"{lang}_model_uploader_label"],
        type=["pth", "pt"],
        key="retrieval_model_uploader",
    )
    st.info(TEXTS[f"{lang}_model_requirements_info"])

    if uploaded_retrieval_model is None:
        st.info(TEXTS[f"{lang}_waiting_for_model_info"])
        st.stop()

    retrieval_model_state_key = ("upload",) + build_uploaded_file_state_key(uploaded_retrieval_model)
    retrieval_model_name = uploaded_retrieval_model.name

else:
    if not TRANSFORMERS_AVAILABLE:
        st.error(TEXTS[f"{lang}_transformers_missing_error"])
        st.stop()

    selected_retrieval_recommended_label = st.selectbox(
        TEXTS[f"{lang}_recommended_model_select_label"],
        list(RECOMMENDED_MODEL_CONFIGS.keys()),
        key="retrieval_recommended_model_select",
    )
    retrieval_model_state_key = ("recommended", selected_retrieval_recommended_label)
    retrieval_model_name = selected_retrieval_recommended_label


# Русский: параметры препроцессинга и инференса
# English: preprocessing and inference parameters
st.subheader(TEXTS[f"{lang}_preprocess_header"])

param_col1, param_col2 = st.columns(2)

if retrieval_source == "upload":
    image_size: int | str = param_col1.selectbox(
        TEXTS[f"{lang}_image_size_label"],
        options=[128, 160, 192, 224, 256, 288, 320, 384, 448, 512],
        index=3,
    )
    retrieval_preprocess = build_preprocess(int(image_size))
else:
    image_size = "recommended"
    param_col1.info(TEXTS[f"{lang}_recommended_preprocess_info"])
    try:
        retrieval_preprocess = build_recommended_preprocess_cached(selected_retrieval_recommended_label)
    except Exception as e:
        st.error(TEXTS[f"{lang}_model_load_error"])
        st.write(f"**{TEXTS[f'{lang}_error_details_prefix']}:** `{e}`")
        st.stop()

batch_size = param_col2.selectbox(
    TEXTS[f"{lang}_batch_size_label"],
    options=[4, 8, 16, 32, 64],
    index=3,
)


try:
    if retrieval_source == "upload":
        retrieval_model = load_uploaded_model(
            uploaded_file=uploaded_retrieval_model,
            model_path=RETRIEVAL_MODEL_PATH,
            device=DEVICE,
        )
    else:
        with st.spinner(
            TEXTS[f"{lang}_recommended_model_loading"].format(
                model_name=selected_retrieval_recommended_label
            )
        ):
            retrieval_model = load_recommended_model_cached(
                selected_retrieval_recommended_label,
                str(DEVICE),
            )
except Exception as e:
    st.error(TEXTS[f"{lang}_model_load_error"])
    st.write(f"**{TEXTS[f'{lang}_error_details_prefix']}:** `{e}`")
    st.stop()

st.success(TEXTS[f"{lang}_model_loaded_success"])

embedding_state_key = (
    dataset_state_key,
    retrieval_model_state_key,
    image_size,
    batch_size,
    len(all_dataset_paths),
    str(DEVICE),
)


# Русский: кнопка расчёта эмбеддингов по всему датасету
# English: button for computing embeddings for the whole dataset
compute_embeddings_clicked = st.button(TEXTS[f"{lang}_compute_embeddings_button"])

if compute_embeddings_clicked:
    try:
        with st.spinner(TEXTS[f"{lang}_embeddings_in_progress"]):
            start_time = time.perf_counter()
            embeddings_matrix, embedding_paths = compute_dataset_embeddings(
                model=retrieval_model,
                image_paths=all_dataset_paths,
                preprocess=retrieval_preprocess,
                batch_size=batch_size,
                device=DEVICE,
            )
            total_seconds = time.perf_counter() - start_time
    except Exception as e:
        st.error(TEXTS[f"{lang}_embedding_error"])
        st.write(f"**{TEXTS[f'{lang}_error_details_prefix']}:** `{e}`")
        st.stop()

    images_count = len(embedding_paths)
    images_per_second = images_count / total_seconds if total_seconds > 0 else 0.0
    seconds_per_image = total_seconds / images_count if images_count > 0 else 0.0

    st.session_state["embedding_state_key"] = embedding_state_key
    st.session_state["embedding_result"] = {
        "embeddings_matrix": embeddings_matrix,
        "embedding_paths": embedding_paths,
        "embedding_dim": int(embeddings_matrix.shape[1]),
        "total_seconds": total_seconds,
        "images_per_second": images_per_second,
        "seconds_per_image": seconds_per_image,
    }
    st.session_state["search_result"] = None
    st.session_state["full_eval_result"] = None
    st.session_state["active_result_type"] = None

embedding_result = None
if st.session_state.get("embedding_state_key") == embedding_state_key:
    embedding_result = st.session_state.get("embedding_result")

if embedding_result is None:
    st.info(TEXTS[f"{lang}_compute_embeddings_first_info"])
    st.stop()

st.success(TEXTS[f"{lang}_embeddings_ready_success"])

embedding_metric_col, _ = st.columns([1, 3])
embedding_metric_col.metric(TEXTS[f"{lang}_embedding_dim_metric"], embedding_result["embedding_dim"])

time_col1, time_col2, time_col3 = st.columns(3)
time_col1.metric(TEXTS[f"{lang}_total_seconds_metric"], f"{embedding_result['total_seconds']:.4f}")
time_col2.metric(TEXTS[f"{lang}_images_per_second_metric"], f"{embedding_result['images_per_second']:.4f}")
time_col3.metric(TEXTS[f"{lang}_seconds_per_image_metric"], f"{embedding_result['seconds_per_image']:.6f}")


# Русский: выбор изображения-запроса из текущего просмотра
# English: query image selection from the current view
st.subheader(TEXTS[f"{lang}_query_image"])

query_options = {
    relative_path_for_display(path, display_root): path
    for path in image_paths
}

selected_query_label = st.selectbox(
    TEXTS[f"{lang}_select_query_image_label"],
    list(query_options.keys()),
)

selected_query_path = query_options[selected_query_label]

query_image = load_image_safely(selected_query_path)
if query_image is not None:
    st.image(
        query_image,
        caption=TEXTS[f"{lang}_query_image"],
        width=300,
    )


# Русский: поиск похожих изображений выбранным способом сравнения
# English: similar image search with the selected comparison method
st.subheader(TEXTS[f"{lang}_find_similar_header"])

search_col1, search_col2 = st.columns(2)

similarity_method_label_to_value = {
    TEXTS[f"{lang}_similarity_cosine"]: "cosine",
    TEXTS[f"{lang}_similarity_dot"]: "dot",
    TEXTS[f"{lang}_similarity_euclidean"]: "euclidean",
}

selected_similarity_method_label = search_col1.selectbox(
    TEXTS[f"{lang}_similarity_method_label"],
    list(similarity_method_label_to_value.keys()),
)

similarity_method = similarity_method_label_to_value[selected_similarity_method_label]

if similarity_method == "cosine":
    l2_normalize_before_search = search_col2.checkbox(
        TEXTS[f"{lang}_l2_normalize_label"],
        value=True,
        disabled=True,
    )
else:
    l2_normalize_before_search = search_col2.checkbox(
        TEXTS[f"{lang}_l2_normalize_label"],
        value=True,
    )

max_top_k = max(1, len(embedding_result["embedding_paths"]) - 1)
top_k = st.slider(
    TEXTS[f"{lang}_top_k_slider"],
    min_value=1,
    max_value=max_top_k,
    value=min(10, max_top_k),
)


# Русский: настройка reranking для поиска по одному query
# English: reranking configuration for single-query search
st.subheader(TEXTS[f"{lang}_reranking_header"])

reranking_source_label_to_value = {
    TEXTS[f"{lang}_reranking_source_none"]: "none",
    TEXTS[f"{lang}_model_source_upload"]: "upload",
    TEXTS[f"{lang}_model_source_recommended"]: "recommended",
}

selected_reranking_source_label = st.selectbox(
    TEXTS[f"{lang}_reranking_source_label"],
    list(reranking_source_label_to_value.keys()),
)
reranking_source = reranking_source_label_to_value[selected_reranking_source_label]
reranking_enabled = reranking_source != "none"

reranking_model = None
reranking_preprocess = None
reranking_model_state_key = None
reranking_feature_type = "global"
rerank_top_n = top_k

if reranking_enabled:
    rerank_col1, rerank_col2 = st.columns(2)

    reranking_feature_label_to_value = {
        TEXTS[f"{lang}_reranking_feature_global"]: "global",
        TEXTS[f"{lang}_reranking_feature_patches"]: "patches",
    }
    selected_reranking_feature_label = rerank_col1.selectbox(
        TEXTS[f"{lang}_reranking_feature_type_label"],
        list(reranking_feature_label_to_value.keys()),
    )
    reranking_feature_type = reranking_feature_label_to_value[selected_reranking_feature_label]

    rerank_top_n = rerank_col2.slider(
        TEXTS[f"{lang}_rerank_top_n_slider"],
        min_value=top_k,
        max_value=max_top_k,
        value=min(max(30, top_k), max_top_k),
    )

    if reranking_source == "upload":
        uploaded_reranking_model = st.file_uploader(
            TEXTS[f"{lang}_reranking_model_uploader_label"],
            type=["pth", "pt"],
            key="reranking_model_uploader",
        )

        if uploaded_reranking_model is not None:
            reranking_model_state_key = ("upload",) + build_uploaded_file_state_key(uploaded_reranking_model)
            reranking_preprocess = build_preprocess(int(image_size)) if isinstance(image_size, int) else build_preprocess(224)
            try:
                reranking_model = load_uploaded_model(
                    uploaded_file=uploaded_reranking_model,
                    model_path=RERANKING_MODEL_PATH,
                    device=DEVICE,
                )
            except Exception as e:
                st.error(TEXTS[f"{lang}_model_load_error"])
                st.write(f"**{TEXTS[f'{lang}_error_details_prefix']}:** `{e}`")
                st.stop()
        else:
            reranking_model_state_key = ("upload", None)

    elif reranking_source == "recommended":
        if not TRANSFORMERS_AVAILABLE:
            st.error(TEXTS[f"{lang}_transformers_missing_error"])
            st.stop()

        selected_reranking_recommended_label = st.selectbox(
            TEXTS[f"{lang}_recommended_model_select_label"],
            list(RECOMMENDED_MODEL_CONFIGS.keys()),
            key="reranking_recommended_model_select",
        )
        reranking_model_state_key = ("recommended", selected_reranking_recommended_label)
        reranking_preprocess = build_recommended_preprocess_cached(selected_reranking_recommended_label)

        try:
            with st.spinner(
                TEXTS[f"{lang}_recommended_model_loading"].format(
                    model_name=selected_reranking_recommended_label
                )
            ):
                reranking_model = load_recommended_model_cached(
                    selected_reranking_recommended_label,
                    str(DEVICE),
                )
        except Exception as e:
            st.error(TEXTS[f"{lang}_model_load_error"])
            st.write(f"**{TEXTS[f'{lang}_error_details_prefix']}:** `{e}`")
            st.stop()

    if reranking_model is not None and reranking_preprocess is not None:
        try:
            reranking_input_shape, reranking_output_shapes = infer_model_tensor_shapes(
                model=reranking_model,
                preprocess=reranking_preprocess,
                batch_size=batch_size,
                device=DEVICE,
            )

            tensor_col1, tensor_col2 = st.columns(2)
            tensor_col1.metric(
                TEXTS[f"{lang}_reranking_input_tensor_metric"],
                format_tensor_shape(reranking_input_shape),
            )

            output_shapes_text = "\n".join(
                f"{name}: {format_tensor_shape(shape)}"
                for name, shape in reranking_output_shapes.items()
            )
            tensor_col2.text_area(
                TEXTS[f"{lang}_reranking_output_tensor_metric"],
                value=output_shapes_text,
                height=100,
                disabled=True,
            )
        except Exception:
            st.warning(TEXTS[f"{lang}_reranking_tensor_shapes_unavailable"])

# Русский: ключ состояния поиска по одному изображению
# English: single-query search state key
current_search_state_key = (
    embedding_state_key,
    str(selected_query_path),
    top_k,
    similarity_method,
    l2_normalize_before_search,
    reranking_source,
    reranking_model_state_key,
    reranking_feature_type,
    rerank_top_n,
)

# Русский: ключ состояния полного эксперимента не зависит от выбранного query и reranking
# English: full-evaluation state key does not depend on the selected query and reranking
current_eval_state_key = (
    embedding_state_key,
    top_k,
    similarity_method,
    l2_normalize_before_search,
)


# Русский: две основные операции поиска расположены рядом: query-поиск и полный retrieval-обход датасета
# English: two main search actions are placed side by side: query search and full retrieval dataset evaluation
action_col1, action_col2 = st.columns(2)

find_similar_clicked = action_col1.button(TEXTS[f"{lang}_find_similar_button"])
evaluate_dataset_clicked = action_col2.button(TEXTS[f"{lang}_evaluate_dataset_button"])

# Русский: обработка поиска похожих изображений по одному query
# English: handle single-query similar image search
if find_similar_clicked:
    st.session_state["full_eval_result"] = None
    st.session_state["active_result_type"] = "single_search"

    embedding_paths = embedding_result["embedding_paths"]
    embeddings_matrix = embedding_result["embeddings_matrix"]
    path_to_index = {path: idx for idx, path in enumerate(embedding_paths)}

    if selected_query_path not in path_to_index:
        st.error(TEXTS[f"{lang}_selected_query_not_found_error"])
        st.stop()

    if reranking_enabled and reranking_model is None:
        st.error(TEXTS[f"{lang}_reranking_model_missing_error"])
        st.stop()

    query_index = path_to_index[selected_query_path]

    full_retrieval_indices, full_retrieval_scores, metric_name = rank_all_candidate_images(
        embeddings_matrix=embeddings_matrix,
        query_index=query_index,
        similarity_method=similarity_method,
        l2_normalize_before_search=l2_normalize_before_search,
    )

    used_reranking = False
    full_ranked_indices = full_retrieval_indices
    top_indices = full_retrieval_indices[:top_k]
    top_scores = full_retrieval_scores[:top_k]

    if reranking_enabled and len(full_retrieval_indices) > 0:
        candidate_indices = full_retrieval_indices[:rerank_top_n]

        try:
            with st.spinner(TEXTS[f"{lang}_reranking_in_progress"]):
                reranked_indices, reranked_scores, metric_name = rerank_candidate_indices(
                    reranking_model=reranking_model,
                    reranking_preprocess=reranking_preprocess,
                    query_path=selected_query_path,
                    candidate_indices=candidate_indices,
                    embedding_paths=embedding_paths,
                    batch_size=batch_size,
                    device=DEVICE,
                    reranking_feature_type=reranking_feature_type,
                    similarity_method=similarity_method,
                    l2_normalize_before_search=l2_normalize_before_search,
                )
                remaining_indices = full_retrieval_indices[rerank_top_n:]
                full_ranked_indices = th.cat([reranked_indices, remaining_indices], dim=0)
                top_indices = reranked_indices[:top_k]
                top_scores = reranked_scores[:top_k]
                used_reranking = True
        except ValueError as e:
            if "Patch embeddings" in str(e):
                st.error(TEXTS[f"{lang}_patch_embeddings_missing_error"])
            else:
                st.error(str(e))
            st.stop()
        except Exception as e:
            st.error(TEXTS[f"{lang}_model_load_error"])
            st.write(f"**{TEXTS[f'{lang}_error_details_prefix']}:** `{e}`")
            st.stop()

    metrics = compute_metrics_from_ranked_indices(
        ranked_indices=full_ranked_indices,
        embedding_paths=embedding_paths,
        query_path=selected_query_path,
        path_to_label=path_to_label,
        top_k=top_k,
    )

    st.session_state["search_result"] = {
        "state_key": current_search_state_key,
        "top_indices": top_indices,
        "top_scores": top_scores,
        "metric_name": metric_name,
        "metrics": metrics,
        "used_reranking": used_reranking,
        "reranking_feature_type": reranking_feature_type,
    }

# Русский: обработка полного retrieval-обхода датасета
# English: handle full-dataset retrieval evaluation
if evaluate_dataset_clicked:
    st.session_state["search_result"] = None

    if not classes_found:
        st.warning(TEXTS[f"{lang}_full_eval_requires_classes"])
        st.session_state["full_eval_result"] = None
        st.session_state["active_result_type"] = None
    else:
        with st.spinner(TEXTS[f"{lang}_full_eval_in_progress"]):
            summary_df, query_df = evaluate_full_dataset(
                embeddings_matrix=embedding_result["embeddings_matrix"],
                embedding_paths=embedding_result["embedding_paths"],
                path_to_label=path_to_label,
                top_k=top_k,
                similarity_method=similarity_method,
                l2_normalize_before_search=l2_normalize_before_search,
                model_name=retrieval_model_name,
                image_size=image_size,
                batch_size=batch_size,
                display_root=display_root,
            )

        st.session_state["full_eval_result"] = {
            "state_key": current_eval_state_key,
            "summary_df": summary_df,
            "query_df": query_df,
        }
        st.session_state["active_result_type"] = "full_eval"


# Русский: отображение активного результата: полного обхода или поиска по одному изображению
# English: display the active result: full evaluation or single-query search
active_result_type = st.session_state.get("active_result_type")
full_eval_result = st.session_state.get("full_eval_result")
search_result = st.session_state.get("search_result")
full_eval_is_current = (
    full_eval_result is not None
    and full_eval_result.get("state_key") == current_eval_state_key
)
search_is_current = (
    search_result is not None
    and search_result.get("state_key") == current_search_state_key
)

if active_result_type == "full_eval" and full_eval_is_current:
    summary_df = full_eval_result["summary_df"]
    query_df = full_eval_result["query_df"]

    st.subheader(TEXTS[f"{lang}_experiment_results_header"])
    st.dataframe(summary_df, use_container_width=True)
    st.download_button(
        label=TEXTS[f"{lang}_download_experiment_csv"],
        data=dataframe_to_csv_bytes(summary_df),
        file_name="cbir_experiment_summary.csv",
        mime="text/csv",
    )

    st.subheader(TEXTS[f"{lang}_query_results_header"])
    if query_df.empty or query_df["AP"].notna().sum() == 0:
        st.info(TEXTS[f"{lang}_no_valid_eval_queries"])
    else:
        st.dataframe(query_df, use_container_width=True)
        st.download_button(
            label=TEXTS[f"{lang}_download_query_csv"],
            data=dataframe_to_csv_bytes(query_df),
            file_name="cbir_per_query_metrics.csv",
            mime="text/csv",
        )

elif active_result_type == "single_search" and search_is_current:
    top_indices = search_result["top_indices"]
    top_scores = search_result["top_scores"]
    metric_name = search_result["metric_name"]
    metrics = search_result["metrics"]
    embedding_paths = embedding_result["embedding_paths"]

    if search_result.get("used_reranking"):
        mode = search_result.get("reranking_feature_type", "")
        st.caption(TEXTS[f"{lang}_reranking_used_caption"].format(mode=mode))

    if len(top_indices) == 0:
        st.info(TEXTS[f"{lang}_no_candidate_images_info"])
    else:
        if not classes_found:
            st.info(TEXTS[f"{lang}_metrics_unavailable_info"])
        elif metrics is not None and metrics["relevant_total"] == 0:
            st.info(TEXTS[f"{lang}_not_enough_relevant_items_info"])
        elif metrics is not None:
            st.subheader(TEXTS[f"{lang}_metrics_header"])
            metric_row_1 = st.columns(3)
            metric_row_2 = st.columns(3)

            metric_row_1[0].metric(f"Precision@{top_k}", f"{metrics['precision_at_k']:.4f}")
            metric_row_1[1].metric(f"Recall@{top_k}", f"{metrics['recall_at_k']:.4f}")
            metric_row_1[2].metric("Precision@1", f"{metrics['precision_at_1']:.4f}")
            metric_row_2[0].metric(f"HitRate@{top_k}", f"{metrics['hit_rate_at_k']:.4f}")
            metric_row_2[1].metric("AP", f"{metrics['ap']:.4f}")

        result_columns = st.columns(4)

        for idx, (match_index, score) in enumerate(zip(top_indices.tolist(), top_scores.tolist())):
            match_path = embedding_paths[match_index]
            match_image = load_image_safely(match_path)

            with result_columns[idx % 4]:
                if match_image is not None:
                    st.image(
                        match_image,
                        caption=f"{metric_name}: {score:.4f}",
                        use_container_width=True,
                    )
                    st.caption(
                        TEXTS[f"{lang}_relative_path_caption"].format(
                            path=relative_path_for_display(match_path, display_root)
                        )
                    )
                else:
                    st.warning(f"{TEXTS[f'{lang}_could_not_open_image']}: {match_path.name}")

else:
    st.info(TEXTS[f"{lang}_press_find_button_info"])

Writing app.py


In [ ]:
# Русский: запуск streamlit приложения в фоне, ведение логов
# English: running the streamlit application in the background, logging
!streamlit run app.py &>logs.txt &

In [ ]:
# Русский: создаём публичную ссылку для браузера
# English: creating a public link for the browser
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://condone-causal-pastrami.ngrok-free.dev" -> "http://localhost:8501"
